# FACULTY SOLUTION — Week 5: Currency — transaction, translation, economic exposure
Computes every answer from the canonical CSVs and asserts it against the golden fixtures.

In [ ]:
import pandas as pd, numpy as np, math, json, itertools
d = {s: pd.read_csv(f'../data/{s}.csv') for s in ['fx_shock', 'entity_flows', 'norway_unit_cost', 'existing_hedges', 'forwards_options', 'worked_example_entity']}

In [ ]:

def compute(d):
    fx = d['fx_shock'].set_index('pair'); fl = d['entity_flows'].set_index('flow_id')['annual_musd_pre']
    r = {}
    r['eur_change'] = fx.loc['EURUSD', 'post'] / fx.loc['EURUSD', 'pre'] - 1
    r['nok_usd_value_change'] = fx.loc['USDNOK', 'pre'] / fx.loc['USDNOK', 'post'] - 1
    r['sgd_usd_value_change'] = fx.loc['USDSGD', 'pre'] / fx.loc['USDSGD', 'post'] - 1
    r['norway_benefit_musd'] = -fl['norway_costs'] * r['nok_usd_value_change']
    r['norway_lifting_post'] = d['norway_unit_cost']['value'].iloc[0] * (1 + r['nok_usd_value_change'])
    r['euro_retail_translation_musd'] = fl['euro_retail_receipts'] * r['eur_change']
    r['existing_hedge_gain_musd'] = -d['existing_hedges']['notional_musd'].iloc[0] * r['eur_change']
    r['rot_net_eur_musd'] = fl['rot_product_revenue'] - fl['rot_eur_costs']
    r['rot_natural_hedge_ratio'] = fl['rot_eur_costs'] / fl['rot_product_revenue']
    r['rot_net_impact_musd'] = r['rot_net_eur_musd'] * r['eur_change']
    r['rot_overhedge_loss_musd'] = fl['rot_eur_costs'] * r['eur_change']
    r['sing_impact_musd'] = fl['sing_net_earnings'] * r['sgd_usd_value_change']
    return r


In [ ]:
fx = json.load(open('../fixtures/week5_golden.json'))
r = compute(d)
bad = []
for k, v in fx['results'].items():
    a = r[k]
    ok = (str(a) == str(v)) if isinstance(v, str) else math.isclose(float(a), float(v), rel_tol=1e-3, abs_tol=1e-5)
    print(('OK  ' if ok else 'DIFF'), f'{k:36s}', a)
    if not ok: bad.append(k)
assert not bad, bad
print('FACULTY NOTEBOOK REPRODUCES GOLDEN FIXTURES: PASS')